# ProSIT 2 V2 — Full Model Training, Imbalance Experiments & Deployment Export

**Source:** `train.csv` only via `data/processed_v2/`. **Primary predictors:** `F1_borrowing_primary` (no provisional outcomes, country ID, or lender ID). 

**Evaluation design:** reuse Notebook V2-01's sealed chronological holdout; compare models with **customer-grouped CV** (and optional stratified CV), optimize the decision threshold on development out-of-fold (OOF) predictions, select the model, fit on development rows, and export deployment-ready pipelines. The final holdout is **off by default**; evaluate it only once when the workflow is frozen.

**Important:** No algorithm can be guaranteed to converge on arbitrary data. This notebook increases iteration limits, uses scaled numeric inputs for numerical optimisers, checks `ConvergenceWarning`, logs every failure, and never silently treats a failed fit as a valid result. Linear Regression and tree ensembles do not use the same iterative convergence criterion as logistic/SVM methods. 

**Accuracy caveat:** Default prevalence is low; use PR-AUC, calibration and threshold-specific recall/precision, not accuracy, for default risk. Historical disbursement amount is not an optimal affordable loan limit.

## 0. Setup and run controls
Set `ROOT` to your project folder. The default complete configuration includes all listed syllabus model families; CPU-intensive KNN and kernel SVM runs may take time. To reduce computation without changing evaluation rules, set `RUN_KERNEL_MODELS=False`, with a documented reduction in coverage.

In [ ]:
from pathlib import Path
import json, time, sys, warnings, hashlib, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import joblib, sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.exceptions import ConvergenceWarning
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
    brier_score_loss, log_loss, mean_absolute_error, mean_squared_error, r2_score, precision_recall_curve,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay)

SEED=42
ROOT=Path.cwd() # change if running from notebooks/ instead of project root
PROCESSED=ROOT/'data'/'processed_v2'
AUDITS=ROOT/'audits'/'v2'
SPLITS=ROOT/'experiments_v2'/'splits'
OUT=ROOT/'experiments_v2'/'full_training_v2'
TABLES=OUT/'tables'; PRED=OUT/'predictions'; FIG=OUT/'figures'; LOGS=OUT/'logs'
MODELS=ROOT/'deployment'/'model_artifacts_v2_final'
for p in (TABLES,PRED,FIG,LOGS,MODELS):p.mkdir(parents=True,exist_ok=True)
FEATURE_SET='F1_borrowing_primary'
PROTOCOLS=['grouped'] # optional comparison: ['grouped','stratified']
RUN_KERNEL_MODELS=True
RUN_SAMPLING_EXPERIMENTS=True
RUN_FINAL_HOLDOUT=False # deliberately gated; set True exactly once after freezing model selection
REQUIRE_ALL_FOLDS=True
N_JOBS=2
print('Python',sys.version.split()[0],'sklearn',sklearn.__version__,'project',ROOT.resolve())

## 1. Load V2 source, enforce the feature contract, and reuse the prior holdout
We do **not** recreate the holdout or use lifetime profiles for supervision. Inconsistent ID audits stop the workflow. Any existing earlier V1 model is explicitly incompatible.

In [ ]:
def read_json(p):
    p=Path(p)
    if not p.exists():raise FileNotFoundError(p)
    return json.loads(p.read_text(encoding='utf-8'))

manifest=read_json(PROCESSED/'engineering_manifest_v2.json')
assert manifest['output_schema']=='feature_eng_v2' and manifest['original_customer_ids_preserved'] is True
assert manifest['primary_uses_outcome_history'] is False and manifest.get('input','train.csv only')=='train.csv only'
FEATURES=read_json(PROCESSED/(FEATURE_SET+'.json'))
assert FEATURES==read_json(PROCESSED/'classification_features.json')==read_json(PROCESSED/'regression_features.json')
assert not set(FEATURES)&{'ID','customer_id','row_key','tbl_loan_id','Total_Amount','target','country_id','lender_id','due_date'}
CAT=[c for c in read_json(PROCESSED/'categorical_features.json') if c in FEATURES]
NUM=[c for c in FEATURES if c not in CAT]
for filename in ('identity_mismatches.csv','invalid_customer_ids.csv'):
    p=AUDITS/filename
    if not p.exists():raise FileNotFoundError(f'Mandatory audit absent: {p}')
    if len(pd.read_csv(p)):raise RuntimeError('Data integrity failure — nonempty audit: '+str(p))

def load_task(name):
    df=pd.read_csv(PROCESSED/(name+'_dataset.csv'),dtype={'ID':'string','customer_id':'string','tbl_loan_id':'string','loan_type':'string','country_id':'string','lender_id':'string'})
    df['disbursement_date']=pd.to_datetime(df['disbursement_date'],errors='coerce')
    df['due_date']=pd.to_datetime(df['due_date'],errors='coerce')
    assert df['row_key'].is_unique and df['ID'].is_unique
    assert df.customer_id.str.fullmatch(r'[1-9][0-9]{5}').fillna(False).all()
    assert df.disbursement_date.notna().all()
    assert set(FEATURES).issubset(df.columns)
    for f in NUM:df[f]=pd.to_numeric(df[f],errors='coerce')
    df[NUM]=df[NUM].replace([np.inf,-np.inf],np.nan)
    if name=='classification':assert df.target.isin([0,1]).all()
    else:assert pd.to_numeric(df.Total_Amount).gt(0).all()
    split=pd.read_csv(SPLITS/(name+'_holdout_assignments.csv'),dtype={'ID':'string','customer_id':'string'})
    assert split.row_key.is_unique and split.partition.isin(['development','final_holdout']).all()
    part=df[['row_key','ID','customer_id']].merge(split[['row_key','ID','customer_id','partition']],on='row_key',how='left',validate='one_to_one',suffixes=('','_saved'),indicator=True)
    assert part._merge.eq('both').all() and part.ID.eq(part.ID_saved).all() and part.customer_id.eq(part.customer_id_saved).all()
    assert set(part.row_key)==set(df.row_key)
    df=df.merge(part[['row_key','partition']],on='row_key',validate='one_to_one')
    dev=df[df.partition=='development'].reset_index(drop=True)
    hold=df[df.partition=='final_holdout'].reset_index(drop=True)
    assert len(dev)>0 and len(hold)>0
    assert dev.disbursement_date.max()<hold.disbursement_date.min()
    return dev,hold

cls_dev,cls_hold=load_task('classification')
reg_dev,reg_hold=load_task('regression')
assert cls_dev.target.nunique()==2
print('Features:',len(FEATURES),'categorical:',CAT,'numeric:',len(NUM))
print('Classification:',len(cls_dev),'dev;',len(cls_hold),'sealed holdout; default rate',round(cls_dev.target.mean(),4))
print('Regression:',len(reg_dev),'dev;',len(reg_hold),'sealed holdout')

## 2. Reuse or regenerate *development-only* customer-grouped folds
Grouping prevents the same customer appearing in training and validation. We prioritize **grouped 5-fold** CV for model ranking; optional stratified CV measures optimistic random-split performance. Note: group splits do not replicate a true future-time validation regime. The chronological holdout is retained for final assessment.

In [ ]:
def folds_from_saved(dev,task,protocol):
    path=SPLITS/(task+'_cv_fold_assignments.csv')
    if not path.exists():raise FileNotFoundError(path)
    rows=pd.read_csv(path)
    rows=rows[rows.protocol==protocol]
    if rows.empty:raise ValueError(f'No saved {protocol} folds for {task}')
    indices=pd.Series(np.arange(len(dev)),index=dev.row_key)
    pairs=[]
    for k in sorted(rows.fold.unique()):
        subset=rows[rows.fold==k]
        tr_keys=subset.loc[subset.partition=='train','row_key'].tolist()
        va_keys=subset.loc[subset.partition=='validation','row_key'].tolist()
        assert set(tr_keys).issubset(set(indices.index)) and set(va_keys).issubset(set(indices.index))
        tr=indices.loc[tr_keys].to_numpy(dtype=int);va=indices.loc[va_keys].to_numpy(dtype=int)
        assert len(set(tr)&set(va))==0 and len(tr)>0 and len(va)>0
        if protocol=='grouped': assert not (set(dev.iloc[tr].customer_id)&set(dev.iloc[va].customer_id))
        pairs.append((tr,va))
    assert len(pairs)>=3,f'Not enough saved folds: {task}, {protocol}'
    return pairs
FOLDS={}
for task,dev in [('classification',cls_dev),('regression',reg_dev)]:
    for prot in PROTOCOLS:FOLDS[(task,prot)]=folds_from_saved(dev,task,prot)
    print(task,{p:len(FOLDS[(task,p)]) for p in PROTOCOLS})
assert ('classification','grouped') in FOLDS and ('regression','grouped') in FOLDS

## 3. Pipeline builders and full model families
Numerical imputation + missing indicators; optional scaling for linear, kernel, and KNN models; one-hot encode `loan_type` within each training fold. Training targets never enter preprocessing. For imbalanced models we test class weights and within-fold resampling. We avoid SMOTE on categorical one-hot arrays; random over-/undersampling is supported directly. Keep original class proportions in validation.

In [ ]:
def preprocess(scale=False):
    numeric=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True))]+([('scale',StandardScaler())] if scale else []))
    categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    return ColumnTransformer([('numeric',numeric,NUM),('categorical',categorical,CAT)],remainder='drop',sparse_threshold=0)

def classification_models():
    models={
        'dummy_majority':(DummyClassifier(strategy='most_frequent'),False),
        'logistic':(LogisticRegression(solver='lbfgs',max_iter=4000,tol=1e-4,random_state=SEED),True),
        'logistic_balanced':(LogisticRegression(solver='lbfgs',max_iter=4000,tol=1e-4,class_weight='balanced',random_state=SEED),True),
        'logistic_weight10':(LogisticRegression(solver='lbfgs',max_iter=4000,tol=1e-4,class_weight={0:1,1:10},random_state=SEED),True),
        'decision_tree_pruned':(DecisionTreeClassifier(max_depth=6,min_samples_leaf=30,ccp_alpha=0.0001,random_state=SEED),False),
        'random_forest':(RandomForestClassifier(n_estimators=140,max_depth=12,min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED),False),
        'random_forest_weight10':(RandomForestClassifier(n_estimators=140,max_depth=12,min_samples_leaf=5,class_weight={0:1,1:10},n_jobs=N_JOBS,random_state=SEED),False),
        'hgb':(HistGradientBoostingClassifier(max_iter=180,learning_rate=.06,max_leaf_nodes=15,early_stopping=True,n_iter_no_change=15,random_state=SEED),False),
    }
    if RUN_KERNEL_MODELS:
        models['knn']=(KNeighborsClassifier(n_neighbors=31,weights='distance',n_jobs=N_JOBS),True)
        models['svc_rbf']=(SVC(kernel='rbf',C=1.0,cache_size=300,probability=True,class_weight='balanced',random_state=SEED),True)
    return models

def regression_models():
    m={
        'dummy_mean':(DummyRegressor(strategy='mean'),False),
        'ols':(LinearRegression(),True),
        'ridge':(Ridge(alpha=20.0),True),
        'lasso':(Lasso(alpha=50.0,max_iter=20000,tol=1e-3,selection='cyclic',random_state=SEED),True),
        'decision_tree':(DecisionTreeRegressor(max_depth=8,min_samples_leaf=20,random_state=SEED),False),
        'random_forest':(RandomForestRegressor(n_estimators=120,max_depth=13,min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED),False),
        'hgb':(HistGradientBoostingRegressor(max_iter=180,learning_rate=.06,max_leaf_nodes=15,early_stopping=True,random_state=SEED),False),
        'hgb_log':(TransformedTargetRegressor(regressor=HistGradientBoostingRegressor(max_iter=180,learning_rate=.06,max_leaf_nodes=15,early_stopping=True,random_state=SEED),func=np.log1p,inverse_func=np.expm1,check_inverse=False),False),
        'ridge_log':(TransformedTargetRegressor(regressor=Ridge(alpha=20.0),func=np.log1p,inverse_func=np.expm1,check_inverse=False),True),
    }
    if RUN_KERNEL_MODELS:
        m['knn']=(KNeighborsRegressor(n_neighbors=25,weights='distance',n_jobs=N_JOBS),True)
        m['svr_rbf']=(SVR(kernel='rbf',C=100.0,epsilon=0.2,cache_size=300),True)
    return m

def pipeline_for(task,model):
    est,scale=(classification_models() if task=='classification' else regression_models())[model]
    return Pipeline([('preprocess',preprocess(scale)),('model',clone(est))])
print('Classification:',list(classification_models()))
print('Regression:',list(regression_models()))

## 4. Model comparisons and imbalance treatments
All training uses **fold-only** data. We record each fold immediately to disk so interrupted sessions preserve progress. A convergence warning is a failure needing attention, not a successful fit. Advanced random resampling uses `imbalanced-learn` and only selected promising model families; it never resamples validation.

In [ ]:
# Optional dependency: python -m pip install imbalanced-learn
try:
    from imblearn.over_sampling import RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    IMB_OK=True
except ImportError:
    IMB_OK=False
    if RUN_SAMPLING_EXPERIMENTS:
        print('imbalanced-learn not installed; resampling experiments will be recorded as unavailable.')

# No SMOTE applied on one-hot categories. Ratios are minority/majority after resampling.
SAMPLING={'random_over_010':('over',.10),'random_under_010':('under',.10)} if RUN_SAMPLING_EXPERIMENTS and IMB_OK else {}

def fit_predict(task,model,tr,va,sampler=None):
    pipe=pipeline_for(task,model)
    ycol='target' if task=='classification' else 'Total_Amount'
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always',ConvergenceWarning)
        if sampler:
            # Explicit fitting order: preprocessing fit TRAIN only, resample TRAIN only, then estimator.
            pre=pipe.named_steps['preprocess']
            Xt=pre.fit_transform(tr[FEATURES]);Xv=pre.transform(va[FEATURES])
            kind,ratio=SAMPLING[sampler]
            obj=(RandomOverSampler(sampling_strategy=ratio,random_state=SEED) if kind=='over' else
                 RandomUnderSampler(sampling_strategy=ratio,random_state=SEED))
            # Oversampling target ratio must exceed the original minority fraction; undersampling must also be feasible.
            Xt2,yt2=obj.fit_resample(Xt,tr[ycol].to_numpy())
            estimator=pipe.named_steps['model'];estimator.fit(Xt2,yt2)
            pred=estimator.predict_proba(Xv)[:,1]
            train_counts=np.bincount(np.asarray(yt2,dtype=int),minlength=2).tolist()
        else:
            pipe.fit(tr[FEATURES],tr[ycol])
            pred=(pipe.predict_proba(va[FEATURES])[:,1] if task=='classification' else pipe.predict(va[FEATURES]))
            train_counts=np.bincount(tr.target.astype(int),minlength=2).tolist() if task=='classification' else []
    convergence=[str(x.message) for x in captured if issubclass(x.category,ConvergenceWarning)]
    if convergence:raise RuntimeError('ConvergenceWarning: '+' | '.join(convergence[:3]))
    return np.asarray(pred,dtype=float),train_counts

def class_metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    h=(p>=.5).astype(int)
    return {'pr_auc':average_precision_score(y,p) if len(np.unique(y))>1 else np.nan,
            'roc_auc':roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
            'precision_05':precision_score(y,h,zero_division=0),'recall_05':recall_score(y,h,zero_division=0),
            'f1_05':f1_score(y,h,zero_division=0),'brier':brier_score_loss(y,p),
            'balanced_accuracy':balanced_accuracy_score(y,h)}

def reg_metrics(y,p):
    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
    if not np.isfinite(p).all():raise ValueError('Non-finite regression prediction')
    clipped=np.maximum(p,0)
    return {'mae_kes':mean_absolute_error(y,clipped),'rmse_kes':np.sqrt(mean_squared_error(y,clipped)),
        'r2':r2_score(y,clipped),'mae_log':mean_absolute_error(np.log1p(y),np.log1p(clipped)),
        'rmse_log':np.sqrt(mean_squared_error(np.log1p(y),np.log1p(clipped))),
        'negative_raw_count':int((p<0).sum())}

def jobs_for(task):
    jobs=[(name,None) for name in (classification_models() if task=='classification' else regression_models())]
    if task=='classification':
        jobs += [(name,strategy) for name in ['logistic','random_forest'] for strategy in SAMPLING]
    return jobs

def run_task(task,dev):
    ycol='target' if task=='classification' else 'Total_Amount'
    metric_rows=[];prediction_rows=[];error_rows=[]
    for protocol in PROTOCOLS:
        folds=FOLDS[(task,protocol)]
        for model,sampler in jobs_for(task):
            run_id=model+('__'+sampler if sampler else '')
            print(f'\n{task} / {protocol} / {run_id}',flush=True)
            for fold_id,(tridx,vaidx) in enumerate(folds,1):
                tr=dev.iloc[tridx];va=dev.iloc[vaidx]
                if task=='classification' and (tr.target.nunique()!=2 or va.target.nunique()!=2):
                    error_rows.append({'task':task,'protocol':protocol,'model':run_id,'fold':fold_id,'error':'Single-class fold'});continue
                started=time.perf_counter()
                try:
                    prediction,counts=fit_predict(task,model,tr,va,sampler)
                    scored=class_metrics(va[ycol],prediction) if task=='classification' else reg_metrics(va[ycol],prediction)
                    metric_rows.append({'task':task,'protocol':protocol,'model':run_id,'fold':fold_id,'train_n':len(tr),'val_n':len(va),
                        'runtime_sec':round(time.perf_counter()-started,2),'train_counts_after_sampling':json.dumps(counts),**scored})
                    prediction_rows.extend({'task':task,'protocol':protocol,'model':run_id,'fold':fold_id,'row_key':int(rk),
                        'actual':float(y),'prediction':float(p)} for rk,y,p in zip(va.row_key,va[ycol],prediction))
                    print(f'  fold {fold_id}/{len(folds)} OK, {time.perf_counter()-started:.1f}s',flush=True)
                except Exception as exc:
                    error_rows.append({'task':task,'protocol':protocol,'model':run_id,'fold':fold_id,'error':str(exc)[:500]})
                    print('  FAILED',fold_id,repr(exc),flush=True)
                pd.DataFrame(metric_rows).to_csv(TABLES/(task+'_fold_metrics.csv'),index=False)
                pd.DataFrame(prediction_rows).to_csv(PRED/(task+'_oof_predictions.csv'),index=False)
                pd.DataFrame(error_rows,columns=['task','protocol','model','fold','error']).to_csv(LOGS/(task+'_errors.csv'),index=False)
    if not metric_rows:raise RuntimeError(f'All {task} experiments failed. Review {LOGS}')
    return pd.DataFrame(metric_rows),pd.DataFrame(prediction_rows),pd.DataFrame(error_rows)


## 5. Execute and save all model runs
**This is the compute-intensive section.** RBF SVC/SVR can be expensive on tens of thousands of rows, especially with probability estimation. For the full syllabus comparison they are enabled; disable only if runtime prevents completion, and document exclusions. Every fold logs execution and errors.

In [ ]:
cls_metrics,cls_oof,cls_errors=run_task('classification',cls_dev)
reg_metrics_df,reg_oof,reg_errors=run_task('regression',reg_dev)
print('Completed folds:',len(cls_metrics),'classification;',len(reg_metrics_df),'regression')
print('Failures:',len(cls_errors),'classification;',len(reg_errors),'regression')

## 6. Quality gate and selection using grouped validation only
Model selection uses **pooled validation predictions**, so folds of different sizes do not receive equal artificial weight. Only models completing all requested grouped folds are eligible. The classification primary metric is pooled PR-AUC; regression primary metric is pooled MAE in KES. Additional metrics and fold variability remain visible. Failed and non-converged runs are excluded.

In [ ]:
def summarize(task,metrics,oop):
    results=[]
    ycol='target' if task=='classification' else 'Total_Amount'
    dev=cls_dev if task=='classification' else reg_dev
    for (protocol,model),part in oop.groupby(['protocol','model'],sort=False):
        need=len(FOLDS[(task,protocol)])
        m=metrics[(metrics.protocol==protocol)&(metrics.model==model)]
        if m.fold.nunique()!=need:
            results.append({'protocol':protocol,'model':model,'eligible':False,'completed_folds':m.fold.nunique(),'required_folds':need})
            continue
        # If a row is repeated as validation more than once, this would not be a legitimate OOF pool.
        assert part.row_key.is_unique,(task,protocol,model,'duplicate OOF row')
        y=part.actual.to_numpy();p=part.prediction.to_numpy()
        res=(class_metrics(y,p) if task=='classification' else reg_metrics(y,p))
        results.append({'protocol':protocol,'model':model,'eligible':True,'completed_folds':need,'required_folds':need,
                        'oof_rows':len(part),'validation_prevalence':float(np.mean(y)) if task=='classification' else np.nan,
                        **res})
    summary=pd.DataFrame(results)
    summary.to_csv(TABLES/(task+'_complete_model_comparison.csv'),index=False)
    return summary

cls_summary=summarize('classification',cls_metrics,cls_oof)
reg_summary=summarize('regression',reg_metrics_df,reg_oof)
cls_rank=cls_summary[(cls_summary.protocol=='grouped')&(cls_summary.eligible==True)].sort_values('pr_auc',ascending=False)
reg_rank=reg_summary[(reg_summary.protocol=='grouped')&(reg_summary.eligible==True)].sort_values(['mae_kes','rmse_kes'])
assert len(cls_rank)>0 and len(reg_rank)>0,'No fully completed grouped models eligible for selection'
display(cls_rank);display(reg_rank)
BEST_CLASS=cls_rank.iloc[0]['model']; BEST_REG=reg_rank.iloc[0]['model']
print('Selected by grouped OOF PR-AUC:',BEST_CLASS)
print('Selected by grouped OOF original-scale MAE:',BEST_REG)

# A resampled model requires a fitted preprocessor, sampler and estimator to be saved together.
# For compact deployment, exclude resampling candidates at export unless imbalanced-learn Pipeline is available.
if '__' in BEST_CLASS:
    from imblearn.pipeline import Pipeline as ImbPipeline
    base,method=BEST_CLASS.split('__',1)
    kind,ratio=SAMPLING[method]
    sampler_obj=(RandomOverSampler(sampling_strategy=ratio,random_state=SEED) if kind=='over' else
                 RandomUnderSampler(sampling_strategy=ratio,random_state=SEED))
    est,scale=classification_models()[base]
    final_cls=ImbPipeline([('preprocess',preprocess(scale)),('sampler',sampler_obj),('model',clone(est))])
else:final_cls=pipeline_for('classification',BEST_CLASS)
final_reg=pipeline_for('regression',BEST_REG)

## 7. Development-only threshold selection and research plots
Threshold selection is done on held-out grouped OOF predictions (no final holdout labels). The sweep maximises F2 subject to minimum precision of 10% (configurable); if no threshold meets it, choose best F1 and report the fallback. This threshold is a demo operating point, **not** a cost-justified regulatory credit decision. PR/ROC curves and confusion matrices use OOF predictions.

In [ ]:
chosen=cls_oof[(cls_oof.protocol=='grouped')&(cls_oof.model==BEST_CLASS)].copy()
y=chosen.actual.astype(int).to_numpy();p=chosen.prediction.to_numpy()
assert chosen.row_key.is_unique
min_precision=0.10
rows=[]
for t in np.linspace(.01,.99,99):
    pred=(p>=t).astype(int)
    prec=precision_score(y,pred,zero_division=0);rec=recall_score(y,pred,zero_division=0)
    f1=f1_score(y,pred,zero_division=0)
    f2=5*prec*rec/(4*prec+rec) if 4*prec+rec else 0
    rows.append({'threshold':float(t),'precision':prec,'recall':rec,'f1':f1,'f2':f2,'flagged':int(pred.sum())})
thresholds=pd.DataFrame(rows)
thresholds.to_csv(TABLES/'classification_threshold_sweep.csv',index=False)
feasible=thresholds[thresholds.precision>=min_precision]
if len(feasible):chosen_t=float(feasible.sort_values(['f2','precision'],ascending=False).iloc[0].threshold);rule='max_f2_precision_ge_0.10'
else:chosen_t=float(thresholds.sort_values('f1',ascending=False).iloc[0].threshold);rule='fallback_max_f1_no_precision_feasible'
print('Selected development-only threshold:',chosen_t,'rule:',rule)

plt.figure(figsize=(8,5))
for model in cls_rank.head(5).model:
    q=cls_oof[(cls_oof.protocol=='grouped')&(cls_oof.model==model)]
    a,b,_=precision_recall_curve(q.actual.astype(int),q.prediction)
    plt.plot(b,a,label=f'{model} (AP={average_precision_score(q.actual,q.prediction):.3f})')
plt.xlabel('Recall');plt.ylabel('Precision');plt.title('Grouped OOF precision–recall curves');plt.legend(fontsize=8)
plt.tight_layout();plt.savefig(FIG/'classification_pr_curves.png',dpi=180);plt.savefig(FIG/'classification_pr_curves.pdf');plt.show()

plt.figure(figsize=(8,4));plt.plot(thresholds.threshold,thresholds.precision,label='Precision');plt.plot(thresholds.threshold,thresholds.recall,label='Recall');plt.plot(thresholds.threshold,thresholds.f2,label='F2');plt.axvline(chosen_t,linestyle='--',label='Selected')
plt.xlabel('Decision threshold');plt.ylabel('Metric');plt.legend();plt.title('Grouped OOF threshold analysis')
plt.tight_layout();plt.savefig(FIG/'classification_threshold_sweep.png',dpi=180);plt.show()

fig,ax=plt.subplots(figsize=(5,4));ConfusionMatrixDisplay.from_predictions(y,(p>=chosen_t).astype(int),ax=ax,colorbar=False)
ax.set_title('Grouped OOF confusion matrix (selected threshold)');plt.tight_layout();plt.savefig(FIG/'classification_oof_confusion_matrix.png',dpi=180);plt.show()

pred_df=reg_oof[(reg_oof.protocol=='grouped')&(reg_oof.model==BEST_REG)].copy()
actual=pred_df.actual.to_numpy();estimated=np.maximum(0,pred_df.prediction.to_numpy())
fig,ax=plt.subplots(figsize=(7,4));ax.scatter(np.log1p(estimated),actual-estimated,s=5,alpha=.25)
ax.axhline(0,linestyle='--',color='black');ax.set_xlabel('log1p(predicted KES)');ax.set_ylabel('Residual (actual - predicted KES)')
ax.set_title('Regression grouped OOF residuals');plt.tight_layout();plt.savefig(FIG/'regression_oof_residuals.png',dpi=180);plt.savefig(FIG/'regression_oof_residuals.pdf');plt.show()

fig,ax=plt.subplots(figsize=(8,4));top=reg_rank.head(7).iloc[::-1];ax.barh(top.model,top.mae_kes);ax.set_xlabel('Pooled grouped OOF MAE (KES)');ax.set_title('Regression model comparison');plt.tight_layout();plt.savefig(FIG/'regression_comparison.png',dpi=180);plt.show()

## 8. Fit winning models on development only; save complete pipelines
The classification feature contract includes loan history but excludes historical outcome labels (F1). Save the fitting code, sklearn version, feature lists, class decision rule, and example input rows. We do not retrain on holdout; old V1 artifacts are not overwritten.

In [ ]:
with warnings.catch_warnings(record=True) as issues:
    warnings.simplefilter('always',ConvergenceWarning)
    cutoff=pd.Timestamp(read_json(SPLITS/'holdout_cutoff.json')['date']).normalize()
    cls_eligible=cls_dev[cls_dev.due_date.notna() & (cls_dev.due_date.dt.normalize()<cutoff)].copy()
    assert len(cls_eligible)>100 and cls_eligible.target.nunique()==2, 'Insufficient mature-labelled development rows for final fitting'
    print('Final classifier fitting rows with due date before holdout cutoff:',len(cls_eligible),'of',len(cls_dev))
    final_cls.fit(cls_eligible[FEATURES],cls_eligible.target.astype(int))
    final_reg.fit(reg_dev[FEATURES],reg_dev.Total_Amount.astype(float))
    conv=[str(w.message) for w in issues if issubclass(w.category,ConvergenceWarning)]
    if conv:raise RuntimeError('Final model convergence warning; do not export: '+' | '.join(conv))
cls_file=MODELS/'default_risk_pipeline_v2.joblib'
reg_file=MODELS/'historical_amount_pipeline_v2.joblib'
joblib.dump(final_cls,cls_file,compress=3)
joblib.dump(final_reg,reg_file,compress=3)

meta={'feature_contract_version':'F1_borrowing_primary_v2','source':'train.csv engineered V2',
    'source_sha256':manifest.get('input_sha256'),'feature_set':FEATURE_SET,
    'features':FEATURES,'classification_features':FEATURES,'regression_features':FEATURES,
    'categorical_features':CAT,'numeric_features':NUM,'classification_model':BEST_CLASS,'regression_model':BEST_REG,
    'classification_threshold':chosen_t,'threshold_rule':rule,'validation_protocol':'customer_grouped_5fold',
    'holdout_tested':False,'classifier_training_rows':len(cls_eligible),'outcome_maturity_proxy':'due_date_before_holdout_cutoff','classification_positive_class':1,'regression_currency':'KES',
    'amount_semantics':'historical disbursement, not affordability or automatic limit',
    'python_version':sys.version.split()[0],'sklearn_version':sklearn.__version__,'created_utc':datetime.now(timezone.utc).isoformat()}
(MODELS/'model_metadata_v2.json').write_text(json.dumps(meta,indent=2,default=str),encoding='utf-8')
cls_dev[['row_key']+FEATURES].head(20).to_csv(MODELS/'classification_example_inputs_v2.csv',index=False)
reg_dev[['row_key']+FEATURES].head(20).to_csv(MODELS/'regression_example_inputs_v2.csv',index=False)
# Real inference reload test
lc=joblib.load(cls_file);lr=joblib.load(reg_file)
pp=lc.predict_proba(cls_dev[FEATURES].head(5))[:,1];rr=lr.predict(reg_dev[FEATURES].head(5))
assert np.isfinite(pp).all() and ((pp>=0)&(pp<=1)).all() and np.isfinite(rr).all()
print('Saved & reloaded:',cls_file,reg_file,'threshold',chosen_t)
print('Classification smoke probabilities:',np.round(pp,4))
print('Regression smoke predictions (KES):',np.round(np.maximum(0,rr),2))

## 9. Optional **one-time** final chronological holdout evaluation
Default is `False`. Before enabling, freeze model selection, features and threshold, and verify that the training labels were actually available before the holdout cutoff. Because actual outcome resolution timestamps are missing, this can only be approximated using due dates. **Do not use holdout results to retune models**.

In [ ]:
if RUN_FINAL_HOLDOUT:
    cls_p=final_cls.predict_proba(cls_hold[FEATURES])[:,1]
    reg_p=final_reg.predict(reg_hold[FEATURES])
    cm=class_metrics(cls_hold.target.astype(int).to_numpy(),cls_p)
    flags=(cls_p>=chosen_t).astype(int)
    cm.update({'precision_selected':precision_score(cls_hold.target,flags,zero_division=0),
               'recall_selected':recall_score(cls_hold.target,flags,zero_division=0),
               'f1_selected':f1_score(cls_hold.target,flags,zero_division=0)})
    rm=reg_metrics(reg_hold.Total_Amount.to_numpy(),reg_p)
    result=pd.DataFrame([{'task':'classification','model':BEST_CLASS,**cm},
                         {'task':'regression','model':BEST_REG,**rm}])
    result.to_csv(TABLES/'ONE_TIME_chronological_holdout_metrics.csv',index=False)
    meta['holdout_tested']=True
    (MODELS/'model_metadata_v2.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
    display(result)
else:print('Final holdout remains sealed. No holdout predictions or scores calculated.')

## 10. Reproducible run inventory and limitations
Use the JSON/CSV outputs to defend model choice, convergence diagnostics, and imbalance interventions. **Not guaranteed**: any solver converges automatically; kernel methods may require tuning or more RAM. **Not established**: model calibration, true Ghana performance, affordability, complete outcome timestamp availability, or regulatory compliance.

In [ ]:
run_summary={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'feature_set':FEATURE_SET,'features':FEATURES,
    'selected_classification':BEST_CLASS,'selected_regression':BEST_REG,'threshold':chosen_t,'threshold_rule':rule,
    'cv_protocols':PROTOCOLS,'classification_folds_completed':len(cls_metrics),'regression_folds_completed':len(reg_metrics_df),
    'classification_failures':len(cls_errors),'regression_failures':len(reg_errors),
    'kernel_models_enabled':RUN_KERNEL_MODELS,'resampling_enabled':RUN_SAMPLING_EXPERIMENTS and IMB_OK,
    'final_holdout_evaluated':RUN_FINAL_HOLDOUT,'model_artifact_dir':str(MODELS),
    'sklearn':sklearn.__version__,'source_sha256':manifest.get('input_sha256')}
(OUT/'run_summary.json').write_text(json.dumps(run_summary,indent=2,default=str),encoding='utf-8')
print('COMPLETE:',OUT.resolve());display(pd.DataFrame([run_summary]).T.rename(columns={0:'value'}))
print('Model files:',cls_file.name,reg_file.name)
print('Next: replace old Streamlit artifacts only after model smoke tests and a frozen one-time holdout evaluation.')